# A2.7 · Systems that don't understand agents

**Function A — Security Architecture & Platform → The Identity & Non-Human Identity Engineer**  ·  *Security of AI*

---

**Risk.** Legacy services see only the human's token and grant everything.

**Control.** Token translation plus action-class blocking at the boundary.

**This lab.** Teach a legacy service to refuse agent writes it cannot understand.

| | |
|---|---|
| Open-source tooling | agentgateway, OPA |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A2.7"))

Most systems you must integrate with have no concept of an agent. They have users, and they will happily believe your agent is one.

In [ ]:
from cybercommons import identity

agent = identity.exchange(identity.mint("alice"), "patch-agent", {"repo:write"})

def legacy_system(token):
    """A system that understands only `sub`. Most of them."""
    return {"authenticated_as": token.sub, "audit_line": f"{token.sub} performed write"}

def agent_aware(token):
    return {"authenticated_as": token.sub, "acting": token.actor,
            "chain": token.chain(),
            "audit_line": f"{token.actor} performed write on behalf of {token.sub}"}

print("legacy      :", legacy_system(agent))
print()
print("agent-aware :", agent_aware(agent))

The token carries the truth; the legacy system throws it away. That is the integration problem in one line — and the mitigation is not to fix the legacy system but to keep the chain at the gateway, which *is* agent-aware, and reconcile logs against it afterwards.

### Expect

Both calls authenticate as `alice`. Only the agent-aware handler records `patch-agent` as the actor and produces a truthful audit line.

### Your turn

Write the reconciliation query: given gateway logs with act chains and legacy logs with only `sub`, how would you attribute a legacy log line to the right agent? What has to be true for that join to work?

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A2.7.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*